#### Setup

In [1]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import duckdb
import shutil
from pathlib import Path
from config import RAW_DIR, BRONZE_DIR

CSV_PATH = RAW_DIR / "_RIPS_20260902.csv"
print(f"Archivo fuente: {CSV_PATH}")
print(f"Existe: {CSV_PATH.exists()}")

Archivo fuente: C:\Users\ASUS I7\Documents\rips-colombia-health-segmentation\notebooks\..\data\raw\_RIPS_20260902.csv
Existe: True


#### Leer con encoding correcto

In [2]:
# El archivo tiene encoding UTF-8 mal interpretado. 
# DuckDB lo lee bien con UTF-8 por defecto, pero los textos vienen corruptos.
# Vamos a leer una muestra para diagnosticar.

con = duckdb.connect()

# Leer muestra
df_sample = con.execute(f"""
    SELECT * FROM read_csv_auto('{CSV_PATH}')
    LIMIT 1000
""").fetchdf()

# Ver columnas de texto con problemas
print("Muestra de Departamentos corruptos:")
print(df_sample['Departamento'].unique()[:10])

Muestra de Departamentos corruptos:
['05 - Antioquia' '15 - BoyacÃ¡']


#### Función de corrección de encoding

In [3]:
def fix_encoding(text):
    """
    Corrige problemas de doble encoding en el dataset RIPS.
    El archivo fue interpretado como CP1252 (Windows-1252) 
    cuando en realidad es UTF-8.
    """
    if pd.isna(text):
        return text
    
    # Si no tiene caracteres corruptos tipicos, devolverlo tal cual
    corrupt_chars = ['Ã', 'Â', '‰', 'ï¿½']
    if not any(c in text for c in corrupt_chars):
        return text
    
    # CP1252 es el culpable en la mayoria de los diagnosticos
    try:
        fixed = text.encode('cp1252').decode('utf-8')
        if not any(c in fixed for c in corrupt_chars):
            return fixed
    except:
        pass
    
    # Fallback: latin1 -> utf-8 (para nombres de municipios)
    try:
        fixed = text.encode('latin1').decode('utf-8')
        if not any(c in fixed for c in corrupt_chars):
            return fixed
    except:
        pass
    
    # Si nada funciona, devolver original
    return text


# Pruebas
test_cases = [
    "BogotÃ¡, D.C.",
    "NecoclÃ­", 
    "ESTRÃ‰S",
    "TRASTORNO DE ESTRÃ‰S POSTRAUMATICO",
    "MedellÃ­n",
    "05 - Antioquia"
]

print("=== PRUEBAS ===")
for t in test_cases:
    print(f"{t}  ->  {fix_encoding(t)}")

=== PRUEBAS ===
BogotÃ¡, D.C.  ->  Bogotá, D.C.
NecoclÃ­  ->  Necoclí
ESTRÃ‰S  ->  ESTRÉS
TRASTORNO DE ESTRÃ‰S POSTRAUMATICO  ->  TRASTORNO DE ESTRÉS POSTRAUMATICO
MedellÃ­n  ->  Medellín
05 - Antioquia  ->  05 - Antioquia


#### Aplicar corrección a todo el dataset

In [4]:
con = duckdb.connect()

# Carpeta temporal para chunks
TEMP_DIR = BRONZE_DIR / "temp"
if TEMP_DIR.exists():
    shutil.rmtree(TEMP_DIR)
TEMP_DIR.mkdir(parents=True, exist_ok=True)

chunk_size = 500_000
offset = 0
chunk_num = 0

print("Procesando en chunks de 500,000 filas...")

while True:
    chunk = con.execute(f"""
        SELECT 
            Departamento,
            Municipio,
            Año,
            TipoAtencion,
            Diagnostico,
            NumeroAtenciones
        FROM read_csv_auto('{CSV_PATH}')
        LIMIT {chunk_size}
        OFFSET {offset}
    """).fetchdf()
    
    if len(chunk) == 0:
        break
    
    # Corregir encoding
    for col in ['Departamento', 'Municipio', 'TipoAtencion', 'Diagnostico']:
        chunk[col] = chunk[col].apply(fix_encoding)
    
    # Parsear codigos DIVIPOLA
    chunk['cod_departamento'] = chunk['Departamento'].str.extract(r'^(\d+)')[0]
    chunk['nombre_departamento'] = chunk['Departamento'].str.extract(r'^\d+\s*-\s*(.*)')[0]
    chunk['cod_municipio'] = chunk['Municipio'].str.extract(r'^(\d+)')[0]
    chunk['nombre_municipio'] = chunk['Municipio'].str.extract(r'^\d+\s*-\s*(.*)')[0]
    
    # Flags NO DEFINIDO
    chunk['es_no_definido'] = chunk['Diagnostico'] == "1 - NO DEFINIDO"
    chunk['es_municipio_no_definido'] = chunk['Municipio'].str.contains('NO DEFINIDO|No Definido', case=False, na=False)
    chunk['es_departamento_no_definido'] = chunk['Departamento'].str.contains('NO DEFINIDO|No Definido', case=False, na=False)
    
    # Guardar chunk temporal (libera RAM inmediatamente)
    temp_file = TEMP_DIR / f"chunk_{chunk_num:03d}.parquet"
    chunk.to_parquet(temp_file, index=False, compression='snappy')
    del chunk
    
    offset += chunk_size
    chunk_num += 1
    print(f"  Chunk {chunk_num:03d} guardado | Total procesado: {offset:,}")

print(f"\nTotal chunks: {chunk_num}")

Procesando en chunks de 500,000 filas...
  Chunk 001 guardado | Total procesado: 500,000
  Chunk 002 guardado | Total procesado: 1,000,000
  Chunk 003 guardado | Total procesado: 1,500,000
  Chunk 004 guardado | Total procesado: 2,000,000


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  Chunk 005 guardado | Total procesado: 2,500,000


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  Chunk 006 guardado | Total procesado: 3,000,000


...

  Chunk 075 guardado | Total procesado: 37,500,000


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  Chunk 076 guardado | Total procesado: 38,000,000


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Total chunks: 76


#### Unir chunks + eliminar duplicados + exportar final

In [5]:
# Leer todos los chunks temporales
print("Leyendo chunks y uniendo...")
temp_files = sorted(TEMP_DIR.glob("chunk_*.parquet"))

dfs = []
for f in temp_files:
    dfs.append(pd.read_parquet(f))

df_clean = pd.concat(dfs, ignore_index=True)
print(f"Registros totales: {len(df_clean):,}")

# Eliminar duplicados exactos
duplicados = df_clean.duplicated().sum()
print(f"Duplicados exactos: {duplicados:,}")
df_clean = df_clean.drop_duplicates()
print(f"Sin duplicados: {len(df_clean):,}")

# Guardar Parquet final
BRONZE_DIR.mkdir(parents=True, exist_ok=True)
parquet_path = BRONZE_DIR / "rips_limpio.parquet"
df_clean.to_parquet(parquet_path, index=False, compression='snappy')

# Limpiar temporales
shutil.rmtree(TEMP_DIR)

print(f"\nParquet final: {parquet_path}")
print(f"Tamanio: {parquet_path.stat().st_size / (1024**3):.2f} GB")

Leyendo chunks y uniendo...
Registros totales: 38,000,000
Duplicados exactos: 12,999,158
Sin duplicados: 25,000,842

Parquet final: C:\Users\ASUS I7\Documents\rips-colombia-health-segmentation\notebooks\..\data\bronze\rips_limpio.parquet
Tamanio: 0.09 GB


#### Verificación final

In [6]:
df_test = pd.read_parquet(parquet_path)

print("=== VERIFICACION ===")
print(f"Filas: {len(df_test):,}")
print(f"Columnas: {list(df_test.columns)}")

print(f"\nDepartamentos (muestra):")
print(df_test['nombre_departamento'].unique()[:5])

print(f"\nMunicipios (muestra):")
print(df_test['nombre_municipio'].unique()[:5])

print(f"\nDiagnosticos con ESTR:")
muestra = df_test[df_test['Diagnostico'].str.contains('ESTR', na=False)]['Diagnostico'].unique()[:3]
for d in muestra:
    print(f"  - {d}")

print(f"\nFlags NO DEFINIDO:")
print(f"  Diagnostico: {df_test['es_no_definido'].sum():,}")
print(f"  Municipio: {df_test['es_municipio_no_definido'].sum():,}")
print(f"  Departamento: {df_test['es_departamento_no_definido'].sum():,}")

=== VERIFICACION ===
Filas: 25,000,842
Columnas: ['Departamento', 'Municipio', 'Año', 'TipoAtencion', 'Diagnostico', 'NumeroAtenciones', 'cod_departamento', 'nombre_departamento', 'cod_municipio', 'nombre_municipio', 'es_no_definido', 'es_municipio_no_definido', 'es_departamento_no_definido']

Departamentos (muestra):
['Antioquia' 'Boyacá' 'Atlántico' None 'Bogotá, D.C.']

Municipios (muestra):
['Copacabana' 'Pueblorrico' 'Belmira' 'Necoclí' 'Anorí']

Diagnosticos con ESTR:
  - F431 - TRASTORNO DE ESTRÉS POSTRAUMATICO
  - O331 - ATENCION MATERNA POR DESPROPORCION DEBIDA A ESTRECHEZ GENERAL DE LA PELVIS
  - I771 - ESTRECHEZ ARTERIAL

Flags NO DEFINIDO:
  Diagnostico: 8,815
  Municipio: 168,137
  Departamento: 168,137
